# Waymo jobs classification test

# 1. Extract data from API and pour to structured data

In [160]:
# Install dependencies from requirements.txt (pinned versions):
# pip install -r requirements.txt
# Or from this notebooks/ folder:
# %pip install -r ../requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import requests
import pandas as pd
from groq import Groq
import json
import html
import re

for parent in [Path.cwd(), *Path.cwd().parents]:
    env_path = parent / "backend" / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        break

In [162]:
# Define waymo career api
WAYMO_CAREER_URL = "https://boards-api.greenhouse.io/v1/boards/waymo/jobs?content=true"
WAYMO_SALARY_URL = "https://www.levels.fyi/companies/waymo/salaries.md"
response = requests.get(WAYMO_CAREER_URL, timeout=30)
response.raise_for_status()

data = response.json()

list_jobs = pd.DataFrame(data["jobs"])
total_jobs = len(list_jobs)
print(total_jobs)

396


In [163]:
# Check columns
print(list_jobs.columns)

Index(['absolute_url', 'data_compliance', 'internal_job_id', 'location',
       'metadata', 'id', 'updated_at', 'requisition_id', 'title',
       'company_name', 'first_published', 'language', 'application_deadline',
       'content', 'departments', 'offices', 'ai_disclaimer',
       'include_ai_disclaimer', 'ai_opt_out_request_url', 'education'],
      dtype='object')


In [164]:
# First job record
first_job = list_jobs.iloc[0]
print(first_job)

absolute_url              https://careers.withwaymo.com/jobs?gh_jid=8049315
data_compliance           [{'type': 'gdpr', 'requires_consent': False, '...
internal_job_id                                                     3488685
location                                 {'name': 'Mountain View, CA, USA'}
metadata                  [{'id': 140578, 'name': 'Job Group', 'value': ...
id                                                                  8049315
updated_at                                        2026-08-05T20:35:37-04:00
requisition_id                                                         5248
title                                                            Accountant
company_name                                                          Waymo
first_published                                   2026-07-31T14:49:37-04:00
language                                                                 en
application_deadline                                                   None
content     

In [165]:
# Check random content of job post
print(first_job["content"])

&lt;div class=&quot;content-intro&quot;&gt;&lt;p&gt;Waymo is an autonomous driving technology company with the mission to be the world&#39;s most trusted driver. Since its start as the Google Self-Driving Car Project in 2009, Waymo has focused on building the Waymo Driver—The World&#39;s Most Experienced Driver™—to improve access to mobility while saving thousands of lives now lost to traffic crashes. The Waymo Driver powers Waymo’s fully autonomous ride-hail service and can also be applied to a range of vehicle platforms and product use cases. The Waymo Driver has provided over ten million rider-only trips, enabled by its experience autonomously driving over 100 million miles on public roads and tens of billions in simulation across 15+ U.S. states.&lt;/p&gt;&lt;/div&gt;&lt;p&gt;Our Driver may be autonomous, but Waymo&#39;s finances are steered by experts like you. The Finance and Accounting group manages all aspects of our finances and serves as trusted advisors for all our strategic

# 2. Cluster job contents to cluster

In [166]:
# Defining the taxanomy structure from job 1
api_key = os.getenv("BAZAARLINK_API_KEY")
if not api_key:
    raise RuntimeError("Set BAZAARLINK_API_KEY in your environment before running this cell.")

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError("Set GROQ_API_KEY in backend/.env or your environment before running the Groq cell.")


In [167]:
job_prompt = f"""You are analyzing a single job posting from an autonomous
vehicle (AV) technology company, as part of a research project mapping
skills demand across the AV industry.
 
Below is the job context. Based ONLY on the information given, extract:
 
1. A short "role profile" name for this specific job (e.g. "Perception
   Engineer", "Cloud Infrastructure Engineer", "Test/QA Engineer",
   "Technical Product Manager") — do not force it into a predefined
   AV-stack category if it doesn't fit (e.g. cloud, test, product,
   business roles are all valid).
2. A list of specific technical skills, tools, and technologies
   mentioned (e.g. "LiDAR", "ROS2", "Kubernetes", "sensor fusion",
   "Gaussian splatting").
3. The general functional area this role likely belongs to within the
   company (e.g. "Onboard / Vehicle Software", "Off-board / Cloud
   Platform", "Business / Product", "Corporate / Support").
 
Respond ONLY in this JSON format, no extra text:
{{
  "company": "...",
  "title": "...",
   "career_page_url": "...",
  "role_profile": "...",
  "skills": ["...", "..."],
  "functional_area": "..."
}}
"""

In [168]:
# Defining the taxanomy structure from job 2
second_job = list_jobs.iloc[1]

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": job_prompt + f""" --- JOB URL --- {second_job['absolute_url']} --- JOB CONTEXT --- {second_job['content']}""",
        }
    ],
)

print(response.choices[0].message.content)

{
  "company": "Waymo",
  "title": "Account Executive - Waymo for Business",
  "career_page_url": "https://careers.withwaymo.com/jobs?gh_jid=8036812",
  "role_profile": "Account Executive (Waymo for Business)",
  "skills": [
    "B2B tech sales",
    "Account management",
    "Salesforce CRM",
    "Outbound pipeline generation",
    "Discovery & demo",
    "Deal closing",
    "Negotiation",
    "Communication with C‑level executives",
    "Stakeholder engagement",
    "Security, legal and procurement navigation",
    "Consumption‑based pricing",
    "Adaptability to evolving playbooks",
    "Cross‑functional collaboration",
    "Product roadmap feedback"
  ],
  "functional_area": "Business / Product"
}


In [169]:
# Testing with GROq model
groq_client = Groq(api_key=groq_api_key)
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": job_prompt + f""" --- JOB URL --- {first_job['absolute_url']} --- JOB CONTEXT --- {first_job['content']}""",
        }
    ]
)

print(response.choices[0].message.content)

{
  "company": "Waymo",
  "title": "Senior Accountant, Technical Accounting",
  "career_page_url": "https://careers.withwaymo.com/jobs?gh_jid=8049315",
  "role_profile": "Technical Accounting Specialist",
  "skills": [
    "Certified Public Accountant (CPA)",
    "US GAAP",
    "SAP Cloud",
    "SAP systems implementation",
    "month‑end close",
    "quarter‑end close",
    "financial statements",
    "financial analytics",
    "flux analysis",
    "technical accounting research",
    "process design and stabilization",
    "automation of accounting processes",
    "cross‑functional collaboration"
  ],
  "functional_area": "Finance and Accounting"
}



In [170]:
response_2 = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": job_prompt + f""" --- JOB URL --- {second_job['absolute_url']} --- JOB CONTEXT --- {second_job['content']}"""
        }
    ]
)

print(response_2.choices[0].message.content)

{
  "company": "Waymo",
  "title": "Account Executive – Corporate Mobility",
  "career_page_url": "https://careers.withwaymo.com/jobs?gh_jid=8036812",
  "role_profile": "Business Development & Partnerships Sales Executive",
  "skills": [
    "Salesforce",
    "CRM"
  ],
  "functional_area": "Business / Partnerships"
}



In [171]:
third_job = list_jobs.loc[list_jobs["id"] == 7474033]

if third_job.empty:
    raise ValueError("No job found for id 7474033")

third_job = third_job.iloc[0]

response_3 = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": job_prompt + f""" --- JOB URL --- {third_job['absolute_url']} --- JOB CONTEXT --- {third_job['content']}"""
        }
    ]
)

print(response_3.choices[0].message.content)

{"company":"Waymo","title":"Machine Learning Research Engineer (RL & Generative AI)","career_page_url":"https://careers.withwaymo.com/jobs?gh_jid=7474033","role_profile":"ML Research Engineer","skills":["Reinforcement Learning","Generative Models (LLM/VLM)","Deep Learning","Sequence Modeling","RLHF","Python","JAX","TensorFlow","Large-Scale Distributed Training","Data Processing","Simulation Platforms","AI Metrics Evaluation"],"functional_area":"Research & Development - Machine Learning"}


Based on some analysis, it's clear that LLM can categorize the the categories for each job, but how it will behave on the large scale is still under the qesution

# 3. Extract salary ranges for relevant Waymo jobs

This section extracts salary ranges from each job description and filters to technical AV-relevant roles for reporting.

In [ ]:
salary_range_pattern = re.compile(
    r"([\$\u00a3\u20ac])\s*([0-9]{2,3}(?:,[0-9]{3})+)\s*(?:-|\u2013|\u2014)\s*([\$\u00a3\u20ac])?\s*([0-9]{2,3}(?:,[0-9]{3})+)\s*([A-Z]{3})?"
)

single_salary_pattern = re.compile(
    r"([\$\u00a3\u20ac])\s*([0-9]{2,3}(?:,[0-9]{3})+)\s*([A-Z]{3})"
)

single_salary_loose_pattern = re.compile(
    r"([\$\u00a3\u20ac])\s*([0-9]{2,3}(?:,[0-9]{3})+)(?:\s*([A-Z]{3}))?"
)

symbol_to_currency = {"$": "USD", "\u00a3": "GBP", "\u20ac": "EUR"}


def extract_salary_from_text(plain_text: str) -> dict:
    if not isinstance(plain_text, str):
        return {}

    match = salary_range_pattern.search(plain_text)
    if match:
        return {
            "salary_min": int(match.group(2).replace(",", "")),
            "salary_max": int(match.group(4).replace(",", "")),
            "salary_currency": match.group(5) or symbol_to_currency.get(match.group(1), "UNKNOWN"),
            "salary_snippet": match.group(0),
        }

    single_match = single_salary_pattern.search(plain_text)
    if single_match:
        amount = int(single_match.group(2).replace(",", ""))
        return {
            "salary_min": amount,
            "salary_max": amount,
            "salary_currency": single_match.group(3) or symbol_to_currency.get(single_match.group(1), "UNKNOWN"),
            "salary_snippet": single_match.group(0),
        }

    loose_match = single_salary_loose_pattern.search(plain_text)
    if loose_match:
        amount = int(loose_match.group(2).replace(",", ""))
        return {
            "salary_min": amount,
            "salary_max": amount,
            "salary_currency": loose_match.group(3) or symbol_to_currency.get(loose_match.group(1), "UNKNOWN"),
            "salary_snippet": loose_match.group(0),
        }

    return {}


def extract_salary_fields(content_html: str) -> pd.Series:
    plain_text = html.unescape(re.sub(r"<[^>]+>", " ", content_html))
    plain_text = re.sub(r"\s+", " ", plain_text).strip()

    extracted = extract_salary_from_text(plain_text)
    if extracted:
        return pd.Series(
            {
                "salary_min_post": extracted["salary_min"],
                "salary_max_post": extracted["salary_max"],
                "salary_currency_post": extracted["salary_currency"],
                "salary_snippet_post": extracted["salary_snippet"],
            }
        )

    return pd.Series(
        {
            "salary_min_post": None,
            "salary_max_post": None,
            "salary_currency_post": None,
            "salary_snippet_post": None,
        }
    )


def normalize_title(title: str) -> set:
    normalized = re.sub(r"[^a-z0-9 ]", " ", str(title).lower())
    tokens = [t for t in normalized.split() if len(t) > 2]
    stop_words = {
        "senior",
        "staff",
        "principal",
        "lead",
        "manager",
        "engineer",
        "specialist",
        "associate",
        "intern",
    }
    return {t for t in tokens if t not in stop_words}


def fetch_levels_salary_entries(levels_url: str) -> list:
    entries = []
    try:
        levels_response = requests.get(
            levels_url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        levels_response.raise_for_status()
        levels_text = levels_response.text
    except Exception as exc:
        print(f"Levels.fyi fetch failed: {exc}")
        return entries

    for raw_line in levels_text.splitlines():
        clean_line = html.unescape(re.sub(r"<[^>]+>", " ", raw_line)).strip()
        if not clean_line:
            continue

        title_candidate = clean_line
        salary_blob = clean_line

        if "|" in clean_line:
            columns = [c.strip() for c in clean_line.split("|") if c.strip()]
            if len(columns) >= 2:
                title_candidate = columns[0]
                salary_blob = " ".join(columns[1:])

        extracted = extract_salary_from_text(salary_blob)
        if not extracted:
            continue

        entries.append(
            {
                "title_text": title_candidate,
                "title_tokens": normalize_title(title_candidate),
                "salary_min": extracted["salary_min"],
                "salary_max": extracted["salary_max"],
                "salary_currency": extracted["salary_currency"],
                "salary_snippet": extracted["salary_snippet"],
            }
        )

    return entries


def find_best_levels_salary(job_title: str, levels_entries: list) -> dict:
    job_tokens = normalize_title(job_title)
    if not job_tokens:
        return {}

    best_match = None
    best_score = 0

    for entry in levels_entries:
        overlap = len(job_tokens & entry["title_tokens"])
        contains = int(
            str(job_title).lower() in entry["title_text"].lower()
            or entry["title_text"].lower() in str(job_title).lower()
        )
        score = overlap + contains

        if score > best_score:
            best_score = score
            best_match = entry

    if best_match and best_score >= 2:
        return best_match

    return {}


post_salary_df = list_jobs["content"].apply(extract_salary_fields)
jobs_with_salary = pd.concat([list_jobs, post_salary_df], axis=1)

levels_entries = fetch_levels_salary_entries(WAYMO_SALARY_URL)


def add_salary_with_fallback(row: pd.Series) -> pd.Series:
    if pd.notna(row["salary_min_post"]):
        return pd.Series(
            {
                "salary_min": row["salary_min_post"],
                "salary_max": row["salary_max_post"],
                "salary_currency": row["salary_currency_post"],
                "salary_snippet": row["salary_snippet_post"],
                "salary_source": "job_post",
            }
        )

    matched = find_best_levels_salary(row.get("title", ""), levels_entries)
    if matched:
        return pd.Series(
            {
                "salary_min": matched["salary_min"],
                "salary_max": matched["salary_max"],
                "salary_currency": matched["salary_currency"],
                "salary_snippet": matched["salary_snippet"],
                "salary_source": "levels_fyi",
            }
        )

    return pd.Series(
        {
            "salary_min": None,
            "salary_max": None,
            "salary_currency": None,
            "salary_snippet": None,
            "salary_source": None,
        }
    )


salary_final_df = jobs_with_salary.apply(add_salary_with_fallback, axis=1)
jobs_with_salary = pd.concat([jobs_with_salary, salary_final_df], axis=1)

jobs_with_salary["location_name"] = jobs_with_salary["location"].apply(
    lambda x: x.get("name") if isinstance(x, dict) else None
)

# Filter to technical AV-relevant roles by title keywords.
relevant_title_pattern = (
    r"machine learning|ml|software|autonomy|autonomous|robot|perception|planning|simulation|ai"
)
relevant_jobs = jobs_with_salary[
    jobs_with_salary["title"].str.contains(relevant_title_pattern, case=False, na=False)
]

relevant_jobs_with_salary = relevant_jobs[
    relevant_jobs["salary_min"].notna()
][
    [
        "id",
        "title",
        "location_name",
        "absolute_url",
        "salary_min",
        "salary_max",
        "salary_currency",
        "salary_snippet",
        "salary_source",
    ]
].sort_values(["salary_currency", "salary_min"], ascending=[True, False])

print(f"Total scraped jobs: {len(list_jobs)}")
print(f"Relevant technical jobs by title: {len(relevant_jobs)}")
print(f"Levels.fyi salary rows parsed: {len(levels_entries)}")
print(f"Relevant technical jobs with salary (post + fallback): {len(relevant_jobs_with_salary)}")
print(
    "Source split:",
    relevant_jobs_with_salary["salary_source"].value_counts(dropna=False).to_dict(),
)

relevant_jobs_with_salary.head(20)

Total scraped jobs: 396
Relevant technical jobs by title: 201
Levels.fyi salary rows parsed: 0
Relevant technical jobs with salary (post + fallback): 179
Source split: {'job_post': 179}


,id,title,location_name,absolute_url,salary_min,salary_max,salary_currency,salary_snippet,salary_source
291,7985945,Staff Machine Learning Engineer,"London, UK",https://careers.withwaymo.com/jobs?gh_jid=7985945,163000.0,163000.0,GBP,"£163,000 GBP",job_post
297,6563711,"Staff Machine Learning Engineer, Simulation","London, UK",https://careers.withwaymo.com/jobs?gh_jid=6563711,163000.0,163000.0,GBP,"£163,000 GBP",job_post
298,7947253,"Staff Machine Learning Engineer, Simulation","London, UK",https://careers.withwaymo.com/jobs?gh_jid=7947253,163000.0,163000.0,GBP,"£163,000 GBP",job_post
301,8064716,"Staff Machine Learning Engineer (TLM), Driver ...","London, UK",https://careers.withwaymo.com/jobs?gh_jid=8064716,163000.0,163000.0,GBP,"£163,000 GBP",job_post
304,7943932,Staff Machine Learning Infrastructure Engineer...,"London, UK",https://careers.withwaymo.com/jobs?gh_jid=7943932,163000.0,163000.0,GBP,"£163,000 GBP",job_post
326,6615783,"Staff Software Engineer, Simulation ML Infrast...","London, UK",https://careers.withwaymo.com/jobs?gh_jid=6615783,163000.0,163000.0,GBP,"£163,000 GBP",job_post
290,7493959,Staff Machine Learning Engineer,"London, England, United Kingdom",https://careers.withwaymo.com/jobs?gh_jid=7493959,162000.0,162000.0,GBP,"£162,000 GBP",job_post
61,7474033,Machine Learning Engineer,"London, England, United Kingdom",https://careers.withwaymo.com/jobs?gh_jid=7474033,130000.0,130000.0,GBP,"£130,000 GBP",job_post
152,7961704,Senior Machine Learning Engineer / London (UK-...,"London, UK",https://careers.withwaymo.com/jobs?gh_jid=7961704,129000.0,129000.0,GBP,"£129,000 GBP",job_post
153,7961706,Senior Machine Learning Engineer / London (UK-...,"London, UK",https://careers.withwaymo.com/jobs?gh_jid=7961706,129000.0,129000.0,GBP,"£129,000 GBP",job_post


# 4. Concatenate taxonomy outputs with relevant job salary data

This section combines model-extracted taxonomy fields with salary fields for the jobs already analyzed by the LLM.

In [173]:
def parse_llm_json(content: str) -> dict:
    text = (content or "").strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:].strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start : end + 1])
        raise


classified_records = []
for job_row, resp_obj, source_name in [
    (first_job, response, "response"),
    (second_job, response_2, "response_2"),
    (third_job, response_3, "response_3"),
]:
    parsed = parse_llm_json(resp_obj.choices[0].message.content)
    parsed["id"] = int(job_row["id"])
    parsed["absolute_url"] = job_row.get("absolute_url")
    parsed["source_response"] = source_name
    classified_records.append(parsed)

classified_jobs_df = pd.DataFrame(classified_records)

final_relevant_jobs_report = classified_jobs_df.merge(
    relevant_jobs_with_salary,
    on=["id", "absolute_url"],
    how="left",
    suffixes=("_taxonomy", "_salary"),
)

final_relevant_jobs_report = final_relevant_jobs_report[
    [
        "id",
        "title_taxonomy",
        "company",
        "career_page_url",
        "absolute_url",
        "role_profile",
        "functional_area",
        "skills",
        "location_name",
        "salary_min",
        "salary_max",
        "salary_currency",
        "salary_snippet",
        "salary_source",
        "source_response",
    ]
]

print(f"Combined rows: {len(final_relevant_jobs_report)}")
final_relevant_jobs_report

Combined rows: 3


,id,title_taxonomy,company,career_page_url,absolute_url,role_profile,functional_area,skills,location_name,salary_min,salary_max,salary_currency,salary_snippet,salary_source,source_response
0,8049315,"Senior Accountant, Technical Accounting",Waymo,https://careers.withwaymo.com/jobs?gh_jid=8049315,https://careers.withwaymo.com/jobs?gh_jid=8049315,Technical Accounting Specialist,Finance and Accounting,"[Certified Public Accountant (CPA), US GAAP, S...",NaN,NaN,NaN,NaN,NaN,NaN,response
1,8036812,Account Executive – Corporate Mobility,Waymo,https://careers.withwaymo.com/jobs?gh_jid=8036812,https://careers.withwaymo.com/jobs?gh_jid=8036812,Business Development & Partnerships Sales Exec...,Business / Partnerships,"[Salesforce, CRM]",NaN,NaN,NaN,NaN,NaN,NaN,response_2
2,7474033,Machine Learning Research Engineer (RL & Gener...,Waymo,https://careers.withwaymo.com/jobs?gh_jid=7474033,https://careers.withwaymo.com/jobs?gh_jid=7474033,ML Research Engineer,Research & Development - Machine Learning,"[Reinforcement Learning, Generative Models (LL...","London, England, United Kingdom",130000.0,130000.0,GBP,"£130,000 GBP",job_post,response_3


# 6. Conclusion

This notebook demonstrates an end-to-end prototype for job intelligence extraction from Waymo postings:

- We successfully scraped live job data from the Greenhouse API and structured it into an analyzable DataFrame.
- We used an LLM workflow to classify selected jobs into a consistent taxonomy format (company, title, URL, role profile, skills, and functional area).
- We extracted salary signals from job post content and linked compensation data back to relevant technical roles.
- We combined taxonomy outputs with salary fields into a single report table for downstream analysis.

## Key findings from Sprint 1

- Data coverage: 396 total jobs were collected from the API.
- Role relevance: 201 jobs matched the technical AV-relevant title filter.
- Salary extraction: 179 relevant jobs included parseable salary values directly from job postings.
- Traceability: Results include job URLs, enabling quick validation against source postings.

## Limitations

- Salary parsing depends on text patterns and may miss edge-case formatting.
- LLM outputs are strong on sampled jobs, but large-scale consistency still needs systematic validation.
- Levels.fyi fallback logic is implemented, but in the current run no parseable salary rows were extracted from that source. (since Levels.fyi only support to show the median salary from some certain job titles)

## Next steps

1. Add schema validation and retry logic for LLM JSON outputs at scale.
2. Improve salary normalization across currencies and location contexts.
3. Add source-quality checks for fallback salary providers.
4. Export final tables to CSV/database and build trend dashboards for sprint reporting.